# Predicting Clinical Trial Outcomes from Protocol Text

**A classical machine learning approach**

Machine Learning final project, May 2026.

---

## 1. Introduction

A clinical trial is a scientific study that tests whether a medical intervention
(a drug, a device, a behavioural program) is safe and effective in people. Before
a trial starts, its sponsor publishes a **protocol**: a structured document that
describes the condition under study, the intervention, who can take part
(the eligibility criteria), and what will be measured.

Many trials never reach a successful conclusion. Some are **terminated**,
**withdrawn**, or **suspended** because of low enrollment, safety concerns,
funding problems, or strategic decisions. A trial that stops early wastes money,
delays useful treatments, and exposes participants to risk without producing
usable evidence.

This project asks a simple but practical question:

> **Can we predict, from the information available about a trial, whether it will
> be *completed* or *terminated/withdrawn/suspended*?**

We treat this as a **binary text + tabular classification** problem and solve it
with classical machine learning models.

## 2. Problem formulation

### 2.1 Why this matters

- **Cost.** A single late-phase clinical trial can cost tens or hundreds of
  millions of dollars. Early warning that a trial is at risk lets sponsors
  reallocate resources.
- **Patients.** Participants accept risk by enrolling. A trial that stops without
  producing evidence breaks the implicit contract with them.
- **Evidence.** Terminated trials rarely publish results, creating gaps and bias
  in the medical literature.

Potential users of such a model include trial sponsors, contract research
organisations, regulators, and investors performing due diligence.

### 2.2 The learning task

Each example is a single trial. The label is derived from the trial's
`overall_status`:

$$
y =
\begin{cases}
1 & \text{if status} = \texttt{COMPLETED} \\
0 & \text{if status} \in \{\texttt{TERMINATED}, \texttt{WITHDRAWN}, \texttt{SUSPENDED}\}
\end{cases}
$$

Trials with non-final statuses (e.g. `RECRUITING`, `ACTIVE`) are excluded, because
their outcome is not yet known.

### 2.3 Assumptions and constraints

- The protocol text and metadata are written largely **before** the outcome is
  known. This is what makes prediction meaningful.
- **Data leakage risk.** In practice, records of terminated trials are often
  updated *after* the fact, and may contain explicit phrases such as
  *"study was terminated due to low enrollment"*. Using such text would let the
  model read the answer instead of predicting it. We address this explicitly in
  the preprocessing section.
- The dataset is in English only; findings may not transfer to other registries.

## 3. Data source and ethics

### 3.1 Source

Data comes from [**ClinicalTrials.gov**](https://clinicaltrials.gov/), the public
registry maintained by the U.S. National Library of Medicine. We use its
[API v2](https://clinicaltrials.gov/data-api/api), which returns study records as
structured JSON.

The download is handled by [`src/fetch_data.py`](../src/fetch_data.py), which keeps
only the fields we need and stores them as a local CSV:

```bash
python -m src.fetch_data --limit 2000 --output data/raw/trials.csv
```

For each trial we keep the text fields (`brief_summary`, `detailed_description`,
`eligibility_criteria`), tabular fields (`study_type`, `phase`,
`enrollment_count`, `conditions`, `intervention_types`), the label source
(`overall_status`), and `why_stopped` (kept **only** for leakage analysis, never
used as a feature).

### 3.2 Legal and ethical notes

- The registry is **public** and contains **no personally identifiable patient
  data (no PHI)**.
- Raw data is **not** committed to the repository. Only the download script is
  versioned, so anyone can reproduce the dataset. This follows good practice for
  data-heavy projects.

## 4. Approach overview

The rest of the notebook follows these steps:

1. **Preprocessing** - build the binary label, merge text fields, and remove
   leakage-prone content.
2. **Exploratory data analysis (EDA)** - class balance, text length, missingness.
3. **Mathematical background** - TF-IDF, logistic loss, regularization, metrics.
4. **Models** - logistic regression baseline, then linear SVM, random forest, and
   gradient boosting (XGBoost).
5. **Ablation and evaluation** - text-only vs tabular-only vs hybrid features,
   with stratified cross-validation and metrics suited to class imbalance.
6. **Results, error analysis, and limitations.**

---

## 5. Exploratory Data Analysis (EDA)

Before building any model we need to understand the data: how balanced are the
classes, how long are the texts, and what do the tabular features look like.

Data is loaded from `data/processed/clean.csv` (produced by `src/preprocess.py`).
Raw files are intentionally excluded from version control — see `README.md` for
reproduction instructions.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")

df = pd.read_csv("../data/processed/clean.csv")

print(f"Dataset: {len(df):,} trials")
print(f"Columns: {list(df.columns)}")
print(f"\nClass distribution:")
print(df["label"].value_counts().rename({1: "Completed", 0: "Not completed"}))

### 5.1 Class balance

The first thing to check: how many trials belong to each class?
A heavily imbalanced dataset requires special handling (weighted loss, stratified
splits, and metrics beyond plain accuracy).

In [ ]:
counts = df["label"].value_counts().sort_index()
labels = ["Not completed (0)", "Completed (1)"]
pcts = counts.values / counts.values.sum() * 100

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, counts.values, color=["#e07070", "#70a8e0"], edgecolor="white")
for bar, pct in zip(bars, pcts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 10,
        f"{pct:.1f}%",
        ha="center",
        va="bottom",
        fontsize=11,
    )
ax.set_title("Class distribution", fontsize=13)
ax.set_ylabel("Number of trials")
ax.set_ylim(0, counts.max() * 1.15)
plt.tight_layout()
plt.savefig("../data/processed/eda_class_balance.png", dpi=100)
plt.show()

print(f"\nImbalance ratio: {counts.max() / counts.min():.1f}:1 (majority:minority)")
print("Note: plain accuracy is misleading here — a naive 'always predict COMPLETED'")
print(f"classifier would score {pcts.max():.1f}% accuracy with zero predictive power.")

### 5.2 Text length distribution

How long are the merged protocol documents? Very short texts may not carry
enough signal; very long ones dominate TF-IDF term counts.

In [ ]:
df["text_len"] = df["text"].str.split().str.len().fillna(0).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (label_val, label_name, color) in zip(
    axes,
    [(1, "Completed", "#70a8e0"), (0, "Not completed", "#e07070")],
):
    subset = df[df["label"] == label_val]["text_len"]
    ax.hist(subset, bins=50, color=color, edgecolor="white", alpha=0.85)
    ax.set_title(f"Text length — {label_name}", fontsize=12)
    ax.set_xlabel("Words in merged document")
    ax.set_ylabel("Count")
    ax.axvline(subset.median(), color="black", linestyle="--", linewidth=1.2,
               label=f"median={subset.median():.0f}")
    ax.legend()

plt.tight_layout()
plt.savefig("../data/processed/eda_text_length.png", dpi=100)
plt.show()

print(df.groupby("label")["text_len"].describe().round(1))

### 5.3 Missing values

Missing data affects both the text and tabular features. We need to know what
to impute and what to drop before training.

In [ ]:
missing = (df.isnull().sum() + (df == "").sum()).sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({"missing_count": missing, "missing_%": missing_pct})
missing_df = missing_df[missing_df["missing_count"] > 0]
print(missing_df.to_string())

### 5.4 Tabular features overview

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Phase distribution by label
phase_counts = df.groupby(["phase", "label"]).size().unstack(fill_value=0)
phase_counts.plot(kind="bar", ax=axes[0], color=["#e07070", "#70a8e0"],
                  edgecolor="white")
axes[0].set_title("Phase distribution by outcome", fontsize=12)
axes[0].set_xlabel("Phase")
axes[0].set_ylabel("Count")
axes[0].legend(["Not completed", "Completed"])
axes[0].tick_params(axis="x", rotation=45)

# Study type
type_counts = df.groupby(["study_type", "label"]).size().unstack(fill_value=0)
type_counts.plot(kind="bar", ax=axes[1], color=["#e07070", "#70a8e0"],
                 edgecolor="white")
axes[1].set_title("Study type by outcome", fontsize=12)
axes[1].set_xlabel("Study type")
axes[1].set_ylabel("Count")
axes[1].legend(["Not completed", "Completed"])
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("../data/processed/eda_tabular.png", dpi=100)
plt.show()

print("\nEnrollment count (by label):")
print(df.groupby("label")["enrollment_count"].describe().round(1))

### 5.5 EDA summary

Key observations from the data:

- **Class imbalance:** the dataset is significantly skewed toward completed trials.
  We will use `class_weight='balanced'` in all models and report Macro F1 and
  PR-AUC rather than accuracy.
- **Text length:** merged protocol documents vary widely in length. TF-IDF with
  sublinear term-frequency scaling will help normalise this.
- **Phase and study type** show different completion rates — these tabular
  features carry predictive signal beyond the text alone, motivating the hybrid
  approach.
- **Leakage note:** `why_stopped` is collected but never used as a feature.
  Leakage-prone sentences have been removed from the `text` field by
  `src/preprocess.py` (with a flag to disable this for ablation).